In [0]:
%pip install requests

In [0]:
import requests

all_data = []
limit = 50
skip = 0

while True:
    url = f"https://dummyjson.com/users?limit={limit}&skip={skip}"
    response = requests.get(url).json()
    users = response.get("users", [])
    
    if not users:
        break
    
    all_data.extend(users)
    skip += limit

print(f"Total records fetched: {len(all_data)}")

In [0]:
import json
print(json.dumps(all_data[0], indent=2))

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("id",        IntegerType(), True),
    StructField("firstName", StringType(),  True),
    StructField("lastName",  StringType(),  True),
    StructField("email",     StringType(),  True),
    StructField("phone",     StringType(),  True),
    StructField("age",       IntegerType(), True),
    StructField("gender",    StringType(),  True),
    StructField("address", StructType([
        StructField("address", StringType(), True),
        StructField("city",    StringType(), True),
        StructField("state",   StringType(), True)
    ]))
])

In [0]:
df = spark.createDataFrame(all_data, schema=schema)
display(df)

In [0]:

df = df.select("id", "firstName", "lastName", "email",
               "phone", "age", "gender", "address")

display(df)

In [0]:
from pyspark.sql.functions import col

df_flat = df.select(
    "id",
    "firstName",
    "lastName",
    "email",
    "phone",
    "age",
    "gender",
    col("address.address").alias("street"),
    col("address.city").alias("city"),
    col("address.state").alias("state")
)

display(df_flat)

In [0]:
from pyspark.sql.functions import split

df_flat = df_flat.withColumn(
    "site_address",
    split("email", "@")[1]
)

display(df_flat)

In [0]:
from pyspark.sql.functions import current_date

df_final = df_flat.withColumn("load_date", current_date())

display(df_final)

In [0]:

df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .save("/Volumes/assingment/q2/api_data/")